# **Purpose**

**Track A — Similarity-based baseline (WordNet).**

This notebook produces distractors with **no generative model at all**: for
each question's correct answer, it finds the head noun, looks up its
WordNet synset, and substitutes in a **co-hyponym** (a sibling word under the
same hypernym — e.g. "cat" and "dog" are both hyponyms of "animal"). This is
the classical lexical-relation approach described in the literature review
(Section 2.4, Similarity-Based Approaches) and gives a non-neural floor to
compare the fine-tuned Track B models against.

It scores on the **exact same fixed RACE test split** as the Track B
evaluation notebooks (same `TEST_SIZE`/`SEED`), and saves its predictions to
a JSON file that the matching evaluation notebook consumes — there's no model
to push to the Hub here.

**Runtime:** CPU is enough for this notebook — no GPU needed, no model
weights to download. Internet must still be ON (to load RACE and NLTK's
WordNet corpus).

## **Install dependencies**

In [1]:
!pip install -qU nltk datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 22.4 MB/s eta 0:00:00


## **Load the API Keys and Tokens**

In [2]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")   # must match the exact secret name you set in Kaggle Secrets

os.environ["HF_TOKEN"] = hf_token

## **Download NLP Libraries**

In [3]:
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"   # Kaggle's network proxy blocks NLTK's default fetch otherwise

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)             # newer NLTK tokenizer resource; harmless if unavailable
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)  # newer NLTK POS-tagger resource

True

## **Verify the downloads**

In [4]:
# Verify the resources actually landed — fails fast with a clear message
# instead of a LookupError deep inside the generation loop later.
required_resources = [
    ("corpora/wordnet", "wordnet"),
    ("corpora/omw-1.4", "omw-1.4"),
    ("tokenizers/punkt_tab", "punkt_tab"),
    ("taggers/averaged_perceptron_tagger_eng", "averaged_perceptron_tagger_eng"),
]

missing = []
for path, name in required_resources:
    try:
        nltk.data.find(path)
    except LookupError:
        missing.append(name)

if missing:
    print(f"WARNING: still missing NLTK resources: {missing}. "
          f"Re-run the download cell above, or check that Internet is ON in Kaggle's Settings panel.")
else:
    print("All required NLTK resources are available.")

## **Imports**

In [5]:
import json
import random

from datasets import load_dataset
from nltk import pos_tag, word_tokenize
from nltk.corpus import wordnet as wn

## **Config**

`TEST_SIZE`/`SEED` must match the Track B evaluation notebooks exactly for
the comparison to be fair.

In [6]:
TEST_SIZE = 1500      # keep identical to Track B evaluation notebooks
SEED = 42              # keep identical to Track B evaluation notebooks
MAX_CANDIDATES = 5     # how many co-hyponym candidates to consider per item

OUTPUT_JSON = "/kaggle/working/wordnet_baseline_predictions.json"

## **Load the fixed test split**

RACE gives 3 gold distractors per question, so we first expand the full test
set into individual (context, question, correct answer) → gold-distractor
pairs, then shuffle those pairs with a fixed seed and take the first
`TEST_SIZE`. **Keep `TEST_SIZE`/`SEED` identical to the Track B (T5/BART/Flan-T5)
evaluation notebooks** — that's what makes this baseline directly comparable
to the fine-tuned models.

In [7]:
LETTER_TO_IDX = {"A": 0, "B": 1, "C": 2, "D": 3}

raw = load_dataset("race", "all")
test_examples = raw["test"]

flat_contexts, flat_questions, flat_answers, flat_distractors = [], [], [], []
for ex in test_examples:
    answer_letter = ex["answer"]
    if answer_letter not in LETTER_TO_IDX:
        continue
    correct_idx = LETTER_TO_IDX[answer_letter]
    options = ex["options"]
    if correct_idx >= len(options):
        continue
    correct_answer = options[correct_idx]

    for i, option in enumerate(options):
        if i == correct_idx or not option.strip():
            continue
        flat_contexts.append(ex["article"])
        flat_questions.append(ex["question"])
        flat_answers.append(correct_answer)
        flat_distractors.append(option)

print(f"Total (context, question, answer) -> gold distractor pairs available: {len(flat_distractors)}")

indices = list(range(len(flat_distractors)))
random.Random(SEED).shuffle(indices)
indices = indices[:TEST_SIZE]

contexts = [flat_contexts[i] for i in indices]
questions = [flat_questions[i] for i in indices]
correct_answers = [flat_answers[i] for i in indices]
gold_distractors = [flat_distractors[i] for i in indices]

print(f"Evaluating on {len(gold_distractors)} pairs")

README.md: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/2.08M [00:00<?, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/37.4M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/2.05M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4934 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/87866 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4887 [00:00<?, ? examples/s]

Total (context, question, answer) -> gold distractor pairs available: 14802
Evaluating on 1500 pairs


## **Head-word extraction**

RACE answers are often full phrases ("a type of renewable energy"), not
single words, so WordNet lookup is done on the phrase's **head noun**
(falling back to the last token if no noun is found) rather than the whole
phrase.

In [8]:
def extract_headword(phrase):
    tokens = word_tokenize(phrase)
    if not tokens:
        return phrase
    tagged = pos_tag(tokens)
    nouns = [w for w, t in tagged if t.startswith("NN")]
    return nouns[-1] if nouns else tokens[-1]

## **WordNet co-hyponym candidate generation**

For the head word: take its first noun synset, go up to its hypernym(s), and
collect sibling words (other hyponyms of that hypernym) as candidates. Falls
back to antonyms of the word itself if no co-hyponyms are found.

In [9]:
def wordnet_candidates(word, max_candidates=MAX_CANDIDATES):
    synsets = wn.synsets(word, pos=wn.NOUN) or wn.synsets(word)
    if not synsets:
        return []

    synset = synsets[0]
    candidates = []
    seen = {word.lower()}

    for hypernym in synset.hypernyms():
        for sibling in hypernym.hyponyms():
            if sibling == synset:
                continue
            for lemma in sibling.lemmas():
                name = lemma.name().replace("_", " ")
                if name.lower() not in seen:
                    seen.add(name.lower())
                    candidates.append(name)

    if not candidates:
        for lemma in synset.lemmas():
            for ant in lemma.antonyms():
                name = ant.name().replace("_", " ")
                if name.lower() not in seen:
                    seen.add(name.lower())
                    candidates.append(name)

    return candidates[:max_candidates]

## **Build a distractor by substituting the head word**

If the head word literally appears in the answer phrase, swap just that word
(preserving the rest of the phrase); otherwise fall back to the candidate
word alone.

In [10]:
def build_distractor(correct_answer, headword, candidate):
    lower_answer = correct_answer.lower()
    lower_head = headword.lower()
    idx = lower_answer.find(lower_head)
    if idx == -1:
        return candidate
    return correct_answer[:idx] + candidate + correct_answer[idx + len(headword):]

## **Generate one WordNet-based distractor per test item**

In [11]:
predictions = []
found_candidate_count = 0

for correct_answer in correct_answers:
    headword = extract_headword(correct_answer)
    candidates = wordnet_candidates(headword)

    if candidates:
        found_candidate_count += 1
        prediction = build_distractor(correct_answer, headword, candidates[0])
    else:
        prediction = ""  # no WordNet relation found for this answer

    predictions.append(prediction)

coverage = found_candidate_count / len(correct_answers) * 100
print(f"WordNet found at least one candidate for {found_candidate_count}/{len(correct_answers)} "
      f"items ({coverage:.1f}% coverage)")

WordNet found at least one candidate for 1262/1500 items (84.1% coverage)


## **Save predictions for the evaluation notebook**

In [12]:
with open(OUTPUT_JSON, "w") as f:
    json.dump(
        {
            "method": "wordnet_baseline",
            "test_size": TEST_SIZE,
            "seed": SEED,
            "coverage_pct": coverage,
            "predictions": predictions,
            "references": gold_distractors,
            "correct_answers": correct_answers,
            "questions": questions,
        },
        f,
        indent=2,
    )

print(f"Saved {len(predictions)} predictions to {OUTPUT_JSON}")

Saved 1500 predictions to /kaggle/working/wordnet_baseline_predictions.json


## **Inspect sample generations**

In [13]:
for correct, gold, pred in list(zip(correct_answers, gold_distractors, predictions))[:10]:
    print(f"Correct answer:       {correct}")
    print(f"Gold distractor:      {gold}")
    print(f"Generated distractor: {pred if pred else '(no candidate found)'}")
    print("-" * 80)

Correct answer:       Working or talking with students.
Gold distractor:      Having a basketball game.
Generated distractor: (no candidate found)
--------------------------------------------------------------------------------
Correct answer:       we must know who we are
Gold distractor:      we should know what we do
Generated distractor: we must know who we quarter section
--------------------------------------------------------------------------------
Correct answer:       the high quality education and research and the wide range of courses
Gold distractor:      the convenient traffic
Generated distractor: the high quality education and research and the wide range of coeducation
--------------------------------------------------------------------------------
Correct answer:       they watch TV late
Gold distractor:      they play the computer games late into the night
Generated distractor: they watch radio late
---------------------------------------------------------------------

## **Next steps**

- Run `evaluate_wordnet_baseline_race_distractor.ipynb` to score these
  predictions with the same metric suite used for T5-base/BART-base/Flan-T5-base.
- Try relaxing the head-word extraction (e.g. allow adjectives/verbs, not
  just nouns) to raise WordNet coverage on non-noun-phrase answers.
- Compare `coverage_pct` against the embedding baseline — WordNet's main
  weakness versus embeddings is usually coverage, not quality of the
  candidates it does find.